
# Production Profit Optimizer

### Mathematical Optimization of Production and Pricing Under Uncertain Demand

## 1. Project Overview

Small businesses often need to decide how many units of a product to order and what selling price to set. These decisions are important because production costs must be paid even when some products remain unsold.

This project develops a mathematical model to help businesses explore production and pricing decisions under a limited budget and uncertain customer demand.

The model is designed for businesses that purchase finished products from external manufacturers and then sell them to customers.

A protein candy business is used as the main practical example. However, the mathematical approach can also be applied to other products by changing the input parameters.



## 2. Problem Statement and Objectives

The main goal of this project is to determine the production quantity and selling price that maximize expected profit while respecting business constraints.

The model considers the following factors:

- Available production budget.
- Fixed and variable production costs.
- Minimum order quantity (MOQ).
- A range of possible selling prices.
- Customer demand estimates at two different prices.
- Uncertainty in actual customer demand.

### Research Questions

The project aims to answer four main questions:

1. What production quantities are feasible given the budget and MOQ?
2. Which combination of production quantity and selling price maximizes profit under expected demand?
3. How does uncertainty in customer demand affect expected profit and the probability of financial loss?
4. How sensitive are the results to changes in costs and demand assumptions?

### Project Approach

First, a deterministic mathematical model will be developed to analyze production costs, demand, revenue, and profit.

Next, mathematical optimization will be used to identify the most profitable feasible decisions.

Finally, Monte Carlo simulations will be performed to estimate expected profit and financial risk under uncertain demand.

The results will be validated through mathematical checks, numerical experiments, and visualizations.



## 3. Assumptions and Scope

To keep the mathematical model understandable and manageable, the following assumptions are made:

1. **Single product:** The model analyzes one product at a time. All quantities refer to finished units ready for sale.

2. **External manufacturing:** The business purchases finished products from an external manufacturer rather than producing them itself.

3. **Constant unit cost:** The production cost per unit remains constant within the analyzed order. Quantity discounts are not considered.

4. **Fixed costs:** Fixed costs do not change with the ordered quantity.

5. **Budget and MOQ:** Production decisions must satisfy the available budget and the manufacturer's minimum order quantity.

6. **Price-dependent demand:** Expected customer demand is approximated by a linear function derived from two price-demand estimates provided by the user.

7. **Single sales period:** All sales are evaluated within one period. Unsold products generate no revenue or residual value during that period.

8. **Uncertain demand:** Actual demand may differ from expected demand. The variability is represented by an explicitly stated probability model.

9. **Synthetic data:** The main experiments use illustrative input values rather than verified historical sales data.

10. **Simplified financial model:** Taxes, financing costs, storage costs, and other expenses not explicitly included in the input parameters are outside the model.

### Scope and Limitations

The project focuses on mathematical modeling, constrained profit optimization, numerical experiments, and Monte Carlo simulation.

It does not attempt to forecast actual market demand, predict future business performance, or provide guaranteed financial recommendations.

The results are conditional on the input data and modeling assumptions. Different assumptions may lead to different optimal decisions.



## 4. Mathematical Model

### 4.1. Input Parameters

The model begins with four basic production parameters: the available budget, fixed costs, unit production cost, and minimum order quantity (MOQ).

The values below represent a synthetic example of a small business purchasing finished products from an external manufacturer.

All monetary values are expressed in euros (EUR).


In [27]:

# Basic production parameters
budget = 5000
fixed_cost = 500
unit_cost = 1.50
moq = 1000

print(f"Available budget: {budget} EUR")
print(f"Fixed costs: {fixed_cost} EUR")
print(f"Unit cost: {unit_cost} EUR")
print(f"Minimum order quantity: {moq} units")


Available budget: 5000 EUR
Fixed costs: 500 EUR
Unit cost: 1.5 EUR
Minimum order quantity: 1000 units



### 4.2. Production Cost Model

Total production cost consists of fixed costs and variable costs.

The cost function is:

$$
C(q) = F + cq
$$

where:

- $C(q)$ is the total cost in EUR.
- $F$ is the fixed cost in EUR.
- $c$ is the cost per unit in EUR.
- $q$ is the ordered quantity.

The model assumes that fixed costs and unit costs remain constant within the analyzed order.


In [28]:
# Example: verify the production cost formula
example_quantity = 3000

example_total_cost = fixed_cost + unit_cost * example_quantity

print(f"Cost for {example_quantity} units: {example_total_cost:.2f} EUR")

Cost for 3000 units: 5000.00 EUR



### 4.3. Budget and Minimum Order Quantity Constraints

The business cannot order an unlimited number of units. The order must satisfy two constraints.

**Budget constraint:**

$$
F + cq \leq B
$$

**Minimum order quantity constraint:**

$$
q \geq MOQ
$$

where:

- $B$ is the available budget.
- $F$ is the fixed cost.
- $c$ is the unit production cost.
- $q$ is the ordered quantity.
- $MOQ$ is the manufacturer's minimum order quantity.

The maximum affordable quantity is:

$$
q_{\max} = \left\lfloor \frac{B-F}{c} \right\rfloor
$$

The floor operation ensures that the quantity is a whole number.

A feasible order exists only if:

$$
q_{\max} \geq MOQ
$$

For the example parameters, the feasible quantity range is from 1,000 to 3,000 units.

The maximum affordable quantity is not necessarily the most profitable quantity. The optimal order will be determined later using the profit model.


In [29]:
max_quantity = int((budget - fixed_cost) // unit_cost)

if max_quantity >= moq:
    print(max_quantity)
else:
    print("Insufficient budget to meet MOQ.")

3000


The cost formula is implemented as a reusable Python function, allowing the model to calculate production costs for different order quantities.

In [37]:
def calculate_total_cost(quantity):
    total_cost = fixed_cost + unit_cost * quantity

    return total_cost

assert calculate_total_cost(1000) == 2000
assert calculate_total_cost(3000) == 5000

print("All production cost tests passed successfully!")

All production cost tests passed successfully!



### 4.4. Customer Demand Model

Customer demand is influenced by the selling price. The model assumes a linear relationship between price and expected demand within the analyzed price range.

The demand function is:

$$
D(p) = a + bp
$$

where:
- $D(p)$ is the expected customer demand in units.
- $p$ is the selling price per unit in EUR.
- $a$ is the intercept of the demand function.
- $b$ is the change in expected demand for a 1 EUR increase in price.

The coefficients are calculated from two user-provided price-demand estimates:

$$
b = \frac{D_2-D_1}{p_2-p_1}
$$

$$
a = D_1-bp_1
$$

The two prices must be different to avoid division by zero.

For the synthetic example:
- At 3 EUR, expected demand is 2,000 units.
- At 5 EUR, expected demand is 1,000 units.

Therefore:

$$
b = \frac{1000-2000}{5-3} = -500
$$

$$
a = 2000-(-500 \cdot 3) = 3500
$$

The resulting demand function is:

$$
D(p) = 3500-500p
$$

This relationship is an assumption based on the provided estimates, not a guarantee of actual sales. The model is applied only within the price range defined by the two estimates.


In [57]:
price_1 = 3.0
demand_1 = 2000

price_2 = 5.0
demand_2 = 1000

if price_1 <= 0 or price_2 <= 0:
    raise ValueError("The selling prices must be positive.")

if demand_1 < 0 or demand_2 < 0:
    raise ValueError("Expected demand cannot be negative.")

if price_1 == price_2:
    raise ValueError("The two prices must be different.")

b = (demand_2 - demand_1) / (price_2 - price_1)

if b > 0:
    raise ValueError("Expected demand must not increase as price increases.")

a = demand_1 - (b * price_1)

print(f"Demand intercept (a): {a:.2f}")
print(f"Demand slope (b): {b:.2f}")


Demand intercept (a): 3500.00
Demand slope (b): -500.00


In [58]:
def calculate_demand(price):
    demand = a + (b * price)

    return demand

# Validate the customer demand model
assert calculate_demand(price_1) == demand_1
assert calculate_demand(price_2) == demand_2

print("Customer demand model tests passed successfully!")

Customer demand model tests passed successfully!



#### Input Validation and Model Assumptions

The input parameters are validated before calculating the demand function.

The following conditions must be satisfied:

- Both selling prices must be positive.
- Expected demand values must be non-negative.
- The two selling prices must be different.
- Expected demand must not increase as the selling price increases.

For the linear demand function:

$$
D(p) = a + bp
$$

the final condition requires:

$$
b \leq 0
$$

Invalid inputs raise a `ValueError` with an explanatory message.

These restrictions ensure that the input data is consistent with the assumptions of the selected demand model.

The model is valid only within the price interval defined by the two input estimates. Predictions outside this interval are not considered in the optimization.
